# Advanced Feature Extraction for Time Series

This notebook demonstrates four feature extraction methods that transform
time series into fixed-length vectors for clustering or classification:

| Method | Key idea | Output |
|---|---|---|
| **ROCKET / MiniRocket** | Random convolutional kernels | PPV + max per kernel |
| **U-Shapelets** | Unsupervised shapelet discovery | Shapelet distance transform |
| **Path Signatures** | Iterated integrals over paths | Truncated signature vector |
| **Vision Embeddings** | Image repr → pretrained CNN | Dense feature vector |

We compare all four on the same synthetic dataset using silhouette score.

In [ ]:
import numpy as np
import polars as pl

np.random.seed(42)


def make_dataset(n_per_class: int = 20, length: int = 50) -> pl.DataFrame:
    """Three classes: sine, sawtooth, step."""
    rows = []
    t = np.linspace(0, 2 * np.pi, length)
    for i in range(n_per_class):
        # Class 0 — sine
        rows.append((f"sine_{i}", np.sin(t + np.random.normal(0, 0.3)) + np.random.normal(0, 0.1, length)))
        # Class 1 — sawtooth
        rows.append((f"saw_{i}", (t / (2 * np.pi) * 2 - 1 + np.random.normal(0, 0.1, length))))
        # Class 2 — step
        step = np.where(t > np.pi + np.random.normal(0, 0.3), 1.0, -1.0) + np.random.normal(0, 0.1, length)
        rows.append((f"step_{i}", step))

    records = []
    for sid, vals in rows:
        for j, v in enumerate(vals):
            records.append({"unique_id": sid, "ds": j, "y": float(v)})
    return pl.DataFrame(records)


df = make_dataset()
print(f"Dataset: {df['unique_id'].n_unique()} series, {len(df)} rows")
df.head()

## 1. ROCKET / MiniRocket

ROCKET applies a large number of random convolutional kernels to each
time series and extracts two features per kernel: the proportion of
positive values (PPV) and the maximum value.

In [ ]:
from polars_ts import minirocket_features, rocket_features

# ROCKET: 500 random kernels → 1000 features
rocket_df = rocket_features(df, n_kernels=500, seed=42)
print(f"ROCKET shape: {rocket_df.shape}")
rocket_df.head(3)

In [ ]:
# MiniRocket: deterministic kernels, faster
minirocket_df = minirocket_features(df, n_kernels=500, seed=42)
print(f"MiniRocket shape: {minirocket_df.shape}")
minirocket_df.head(3)

In [ ]:
from sklearn.cluster import KMeans

from polars_ts import silhouette_score

# Cluster ROCKET features with KMeans
feature_cols = [c for c in rocket_df.columns if c != "unique_id"]
X_rocket = rocket_df.select(feature_cols).to_numpy()
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_rocket)

labels_rocket = pl.DataFrame(
    {
        "unique_id": rocket_df["unique_id"],
        "cluster": km.labels_.tolist(),
    }
)

# Evaluate
sil_rocket = silhouette_score(df, labels_rocket)
print(f"ROCKET silhouette score: {sil_rocket:.3f}")

## 2. U-Shapelets

Unsupervised shapelets discover discriminative subsequences directly
from the data, then use their distance transform for clustering.

In [ ]:
from polars_ts import UShapeletClusterer, shapelet_cluster

# Quick shapelet clustering
labels_shapelets = shapelet_cluster(
    df,
    k=3,
    n_shapelets=10,
    seed=42,
)
print(labels_shapelets.head())

sil_shapelets = silhouette_score(df, labels_shapelets)
print(f"U-Shapelet silhouette score: {sil_shapelets:.3f}")

In [ ]:
# Access the discovered shapelets via the class API
clusterer = UShapeletClusterer(n_clusters=3, n_shapelets=10, seed=42)
clusterer.fit(df)

print(f"Discovered {len(clusterer.shapelets_)} shapelets")
for i, s in enumerate(clusterer.shapelets_[:3]):
    print(f"  Shapelet {i}: length={len(s)}, range=[{s.min():.2f}, {s.max():.2f}]")

## 3. Path Signatures

Path signatures capture the sequential structure of a time series via
iterated integrals. Truncated at depth $, they produce a fixed-length
vector that is invariant to reparametrisation.

In [ ]:
from polars_ts.imaging.signature import signature_features

# Depth-3 signature features with time augmentation
sig_df = signature_features(df, depth=3, augmentations=["time"])
print(f"Signature shape: {sig_df.shape}")
sig_df.head(3)

In [ ]:
# Cluster signature features
sig_feature_cols = [c for c in sig_df.columns if c != "unique_id"]
X_sig = sig_df.select(sig_feature_cols).to_numpy()
km_sig = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_sig)

labels_sig = pl.DataFrame(
    {
        "unique_id": sig_df["unique_id"],
        "cluster": km_sig.labels_.tolist(),
    }
)

sil_sig = silhouette_score(df, labels_sig)
print(f"Signature silhouette score: {sil_sig:.3f}")

## 4. Vision Model Embeddings

Convert time series to images (e.g. Gramian Angular Summation Field),
then extract features from a pretrained vision model.

> **Note**: This section requires  and .
> It will be skipped if they are not installed.

In [ ]:
try:
    import torch  # noqa: F401

    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("torch not installed — skipping vision embeddings")

In [ ]:
if HAS_TORCH:
    from polars_ts.imaging.angular import to_gasf
    from polars_ts.imaging.embeddings import extract_vision_embeddings

    # Step 1: Convert to GASF images
    images = to_gasf(df)
    sample_id = list(images.keys())[0]
    print(f"Image for {sample_id!r}: shape={images[sample_id].shape}")

    # Step 2: Extract ResNet18 embeddings
    emb_df = extract_vision_embeddings(images, model="resnet18")
    print(f"Embedding shape: {emb_df.shape}")

    # Step 3: Cluster
    emb_cols = [c for c in emb_df.columns if c != "unique_id"]
    X_emb = emb_df.select(emb_cols).to_numpy()
    km_emb = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_emb)

    labels_emb = pl.DataFrame(
        {
            "unique_id": emb_df["unique_id"],
            "cluster": km_emb.labels_.tolist(),
        }
    )

    sil_emb = silhouette_score(df, labels_emb)
    print(f"Vision embedding silhouette score: {sil_emb:.3f}")
else:
    sil_emb = float("nan")
    print("Skipped (torch not available)")

## 5. Feature Method Comparison

Benchmark table comparing all four methods on the same 60-series dataset.

In [ ]:
import time


def timed_rocket(df):
    t0 = time.perf_counter()
    feat = rocket_features(df, n_kernels=500, seed=42)
    cols = [c for c in feat.columns if c != "unique_id"]
    X = feat.select(cols).to_numpy()
    km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
    labels = pl.DataFrame({"unique_id": feat["unique_id"], "cluster": km.labels_.tolist()})
    sil = silhouette_score(df, labels)
    return sil, time.perf_counter() - t0


def timed_shapelets(df):
    t0 = time.perf_counter()
    labels = shapelet_cluster(df, k=3, n_shapelets=10, seed=42)
    sil = silhouette_score(df, labels)
    return sil, time.perf_counter() - t0


def timed_signatures(df):
    t0 = time.perf_counter()
    feat = signature_features(df, depth=3, augmentations=["time"])
    cols = [c for c in feat.columns if c != "unique_id"]
    X = feat.select(cols).to_numpy()
    km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)
    labels = pl.DataFrame({"unique_id": feat["unique_id"], "cluster": km.labels_.tolist()})
    sil = silhouette_score(df, labels)
    return sil, time.perf_counter() - t0


sil_r, t_r = timed_rocket(df)
sil_s, t_s = timed_shapelets(df)
sil_g, t_g = timed_signatures(df)

comparison = pl.DataFrame(
    {
        "Method": ["ROCKET", "U-Shapelets", "Path Signatures", "Vision (ResNet18)"],
        "Silhouette": [
            round(sil_r, 3),
            round(sil_s, 3),
            round(sil_g, 3),
            round(float(sil_emb), 3) if not np.isnan(sil_emb) else None,
        ],
        "Time (s)": [round(t_r, 2), round(t_s, 2), round(t_g, 2), None],
    }
)
print(comparison)

## Summary

| Method | Best for | Trade-off |
|---|---|---|
| **ROCKET** | Fast baseline, large datasets | Fixed random kernels, no interpretability |
| **U-Shapelets** | Interpretable sub-pattern discovery | Slower, sensitive to shapelet count |
| **Path Signatures** | Capturing sequential order information | Feature dimension grows with depth |
| **Vision Embeddings** | Leveraging pretrained visual representations | Requires GPU for speed, torch dependency |

**Guidance**: Start with ROCKET for a fast baseline. Use U-Shapelets when
interpretability matters. Path signatures excel on multivariate series where
order matters. Vision embeddings are best when the image representation
(recurrence plot, GAF) captures discriminative structure.